# Subject indexing as semantic search


**Goal**. The main goal of this notebook is to build a system for subject indexing. It aims to predict RAMEAU subject headings (a francophone controlled vocabulary used for indexing library resources) for documents, using the documents's descriptions (bibliographic records).

**Approach**. The subject indexing problem is reframed as a semantic search problem. The vector for each subject heading is calculated as the average of the embeddings of documents associated with that subject heading. Documents are indexed by finding the closest subject headings in a semantic (embedding) space

**Nota Bene** :
- In this notebook, the method is applied to RAMEAU, a large francophone subject headings vocabulary. It could be applied to other indexing vocabularies.
- The dataset is a demo one. In real life, our training dataset includes 400,000 bibliographic records of books. This number increases significantly when the French dissertations dataset is included (over 400,000 records).
- This method focuses on concept indexing, where all subjects are represented as nouns ("Butter", "Atheism", etc.). Geographical or chronological indexing would be better served by other approaches, such as Named Entity Recognition (NER).
- It is recommended to run this document in a GPU environment (ex : Google Colab). The embedding of 10, 000 documents will take 10 minutes (vs. 2 hours on CPU).


More information (FR) : 
- [Final report on a hands-on experiment conducted with twelve French academic libraries in 2024](https://fil.abes.fr/2025/04/10/lindexation-rameau-assistee-par-ia-retour-sur-une-experimentation-prometteuse/)
- General overview of the approach (soon)


This demo notebook was developed by the 'labo', a unit of Abes (Agence bibliographique de l'enseignement supérieur (France)).



In [ ]:
#!pip install qdrant_client
!pip install tf-keras # if version compatibility error

In [2]:
'''
!pip install Unidecode
!pip install sentence_transformers
!pip install "gensim==4.2.0"
!pip install "texthero==1.0.5"
!pip install simplemma
!pip install nltk
'''

'\n!pip install Unidecode\n!pip install sentence_transformers\n!pip install "gensim==4.2.0"\n!pip install "texthero==1.0.5"\n!pip install simplemma\n!pip install nltk\n'

## Python import

In [ ]:
import re
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import pickle
from datetime import datetime 
import requests
import io


## Parameters

In [4]:
# Sudoc bibliographic records.
# Small dataset : 10 000 records
# In this dataset, the RAMEAU indexing chains are note preserved : "Lois -- Histoire" => ["Lois", "Histoire"]

train_filename = "sudoc_rameau_sample_10k"


In [5]:

# Embeddings models
#emb_model = "all-MiniLM-L6-v2"
#emb_model = "intfloat/multilingual-e5-small"
emb_model = "intfloat/multilingual-e5-large" #big and performant


In [6]:
# Vector database populated with RAMEAU concepts embeddings
qdrant_dbname = "subjects_db"
qdrant_collectionname = "qdrant_collection_"+ emb_model.replace("/","_")


## Test data

Les données d'évaluation sont un lot d'une centaine de notices qui ont été réindexées par des humains, indexées par différents algorithmes. Ensuite, des humains ont évalué ces diverses indexations selon une grille d'évaluation multicritères (exactitute et précision de chaque sujet associé à un document, complétude et rendondance d'un ensemble de sujets associé à un document).

In [7]:
# From a local file

#test_data = pd.read_csv("test_data.csv") # 100 biblios pour évaluation
#test_data

In [8]:
# From an URL

try:
    test_data = pd.read_csv("https://raw.githubusercontent.com/abes-esr/labo_subject_indexing/refs/heads/develop/test_data.csv")
except Exception as e:
    print(f"Error loading test data: {e}")

test_data

,PPN,TITRE,RESUME
0,000308838,Les sommets de l'État : essai sur l'élite du p...,"u XIXe siècle à nos jours, l'Etat ""fort"" à la ..."
1,00094758X,Le dollar,"La quatrième de couverture indique : ""Quelle e..."
2,003632806,Les intellectuels sous la Ve République : 1958...,"Célèbres, influents, on les voit, on en parle ..."
3,047450037,"Bouddha, bouddhisme","La 4e de couv. indique : ""Ce petit livre répon..."
4,05224170X,Apprendre à aimer les mathématiques : conditio...,"Les entretiens d'élèves et d'enseignants, anal..."
...,...,...,...
95,266197809,Les familles seigneuriales et le domaine de Va...,"Monographie du village de Valence-en-Brie, pré..."
96,26753177X,Algocratie : allons-nous donner le pouvoir aux...,"Aujourd’hui, nos vies se retrouvent sous l’inf..."
97,267884575,Désirs postcapitalistes,Dans cette série de cours donnés avant sa disp...
98,268799458,Le coenseignement en pratique,Présentation et analyse de tous les aspects du...


## Train data (bibliographic records + list of RAMEAU concepts)

In [9]:
# From a local file

#df_data = pd.read_pickle(train_filename+'.pkl') #.iloc[0:100]
#df_data

In [10]:
# From an URL

try:
    df_data = pd.read_csv("https://raw.githubusercontent.com/abes-esr/labo_subject_indexing/refs/heads/develop/sudoc_rameau_sample_10k.csv") #.iloc[0:100]
except Exception as e:
    print(f"Error loading test data: {e}")

df_data

,DESCR,RAMEAU_concepts2
0,Initiation à la linguistique : avec des travau...,"['027236005#Linguistique', '027789853#Étude_et..."
1,"La culture pour vivre, Mort de la culture popu...","['027416593#Politique_culturelle', '027224929#..."
2,"La Longue traque, Le 23 juillet 1945, on repêc...",['027350282#Purges_politiques']
3,"Stigmate : les usages sociaux des handicaps, I...","['029251133#Identité_collective', '027336360#D..."
4,L'agitation paysanne en Russie de 1881 à 1902 ...,['027246086#Révoltes_paysannes']
...,...,...
9995,"Les bananiers diploi͏̈des en culture ""in vitro...","['031281737#Nombre_de_chromosomes', '027231429..."
9996,Contribution à l'étude de l'effet de la grande...,"['027264998#Enseignement', '027264998#Enseigne..."
9997,Contribution à l'étude de l'effet de la grande...,"['027264998#Enseignement', '027264998#Enseigne..."
9998,"De l'initiation éducative, Deux grandes théori...",['027566056#Rites_d#c#initiation']


## Bibliographic records embeddings

In [11]:
# Loading the model
encoder = SentenceTransformer(emb_model, device='cuda')

In [12]:
# Single example

start_time = datetime.now() 

text = "Techniques modernes en gravité et structures des théories des champs conformes holographiques. Nous introduisons un cadre pour quantifier le comportement des matrices aléatoires des CFTs 2d et de la gravité quantique AdS3. Nous présentons une formule de trace de CFT 2d, précisément analogue à la formule de trace de Gutzwiller pour les systèmes quantiques chaotiques, qui provient de la décomposition spectrale SL(2, Z) de la densité d'états primaire de Virasoro. Une analogie avec l'approximation diagonale de Berry nous permet d'extraire des statistiques spectrales de CFTs 2d individuels par un grossissement, et d'identifier les signatures du chaos et de l'universalité des matrices aléatoires. Cela conduit à une condition nécessaire et suffisante pour qu'un CFT 2d présente une rampe linéaire dans son facteur de forme spectral à gros grain. En ce qui concerne la gravité, les trous de ver du tore AdS3 sont clairement interprétés comme des projections diagonales des fonctions de partition au carré des CFT 2d microscopiques. La projection utilise les opérateurs de Hecke. On montre que le trou de ver de Cotler-Jensen de la gravité pure AdS3 est extrême parmi les amplitudes de trou de ver : c'est la complétion minimale du corrélateur de la théorie des matrices aléatoires compatible avec la symétrie de Virasoro et l'invariance SL(2, Z). Nous l'appelons MaxRMT : la réalisation maximale de l'universalité des matrices aléatoires compatible avec les symétries nécessaires. La complétude de la décomposition spectrale SL(2,Z) en tant que formule de trace nous permet de factoriser le vortex de Cotler-Jensen, en extrayant l'objet microscopique ZRMT(τ) du produit à gros grain. Cela permet de capturer les détails du spectre des micro-états des trous noirs BTZ. ZRMT(τ) peut être interprété comme un demi-trou de ver AdS3. Nous discutons de ses implications pour la CFT duale et le bootstrap modulaire à grande charge centrale."
print(encoder.encode(text))

end_time = datetime.now() 
time_difference = (end_time - start_time).total_seconds() * 10**3
print("Execution time of program is: ", time_difference, "ms") 

[ 0.03225657  0.01948298 -0.02346544 ... -0.01478919 -0.04500858
 -0.02291291]
Execution time of program is:  656.697 ms


In [ ]:
start_time = datetime.now() 

# batch processing
batch_size = 5000

def encode_batches(data, encoder, batch_size):
    print(len(data))
    encoded_batches = []
    # How many batches?
    num_batches = int(np.ceil(len(data) / batch_size))
    print(num_batches)

    for i in range(num_batches):
        print(i)
        batch_data = data.iloc[i * batch_size: (i + 1) * batch_size]
        encoded_batch = encoder.encode(batch_data['DESCR'].tolist(), show_progress_bar=True)
        encoded_batches.append(encoded_batch)

    return np.concatenate(encoded_batches)


# Embedding
# Processing time: 10 minutes with a TP4 GPU on Google colab, but around 2 hours with CPU
encoded_articles = encode_batches(df_data, encoder, batch_size)

# New dataframe, with embeddings (vectors)
df_data_with_vectors = df_data.copy()
df_data_with_vectors["bibliographic_vector"] = pd.Series(encoded_articles.tolist())

end_time = datetime.now() 
time_difference = (end_time - start_time).total_seconds() * 10**3
print("Execution time of program is: ", time_difference, "ms") 


In [14]:
encoded_articles

array([[ 0.01090342,  0.00747017, -0.00670591, ..., -0.02522106,
         0.01063932,  0.00556939],
       [ 0.03385548, -0.01818151,  0.00911111, ..., -0.03368218,
        -0.03515344,  0.00835912],
       [ 0.03381812, -0.00851533, -0.0176127 , ..., -0.03842733,
        -0.00758309,  0.02113778],
       ...,
       [ 0.03788098, -0.00934553, -0.04033094, ..., -0.01008554,
        -0.03922381,  0.01270155],
       [ 0.00993038, -0.03091455, -0.01801262, ..., -0.00568836,
         0.01215   ,  0.00985828],
       [ 0.00424739, -0.01851443, -0.03524906, ..., -0.02746824,
        -0.02482826,  0.01134863]], dtype=float32)

In [15]:
df_data_with_vectors.head()

,DESCR,RAMEAU_concepts2,bibliographic_vector
0,Initiation à la linguistique : avec des travau...,"['027236005#Linguistique', '027789853#Étude_et...","[0.010903417132794857, 0.007470165845006704, -..."
1,"La culture pour vivre, Mort de la culture popu...","['027416593#Politique_culturelle', '027224929#...","[0.03385547548532486, -0.018181508406996727, 0..."
2,"La Longue traque, Le 23 juillet 1945, on repêc...",['027350282#Purges_politiques'],"[0.033818118274211884, -0.008515330962836742, ..."
3,"Stigmate : les usages sociaux des handicaps, I...","['029251133#Identité_collective', '027336360#D...","[0.019312692806124687, -0.00716855563223362, -..."
4,L'agitation paysanne en Russie de 1881 à 1902 ...,['027246086#Révoltes_paysannes'],"[0.015167579054832458, -0.025262555107474327, ..."


In [16]:
# select rows with NaN column
df_data_with_vectors[df_data_with_vectors.isna().any(axis=1)]

,DESCR,RAMEAU_concepts2,bibliographic_vector


In [17]:
df_data_with_vectors.dropna(subset = ['bibliographic_vector'], inplace=True)
df_data_with_vectors = df_data_with_vectors.drop(columns=['DESCR'])

In [18]:
df_data_with_vectors

,RAMEAU_concepts2,bibliographic_vector
0,"['027236005#Linguistique', '027789853#Étude_et...","[0.010903417132794857, 0.007470165845006704, -..."
1,"['027416593#Politique_culturelle', '027224929#...","[0.03385547548532486, -0.018181508406996727, 0..."
2,['027350282#Purges_politiques'],"[0.033818118274211884, -0.008515330962836742, ..."
3,"['029251133#Identité_collective', '027336360#D...","[0.019312692806124687, -0.00716855563223362, -..."
4,['027246086#Révoltes_paysannes'],"[0.015167579054832458, -0.025262555107474327, ..."
...,...,...
9995,"['031281737#Nombre_de_chromosomes', '027231429...","[0.010252579115331173, -0.007863505743443966, ..."
9996,"['027264998#Enseignement', '027264998#Enseigne...","[0.027775287628173828, -0.01775376684963703, -..."
9997,"['027264998#Enseignement', '027264998#Enseigne...","[0.03788097947835922, -0.009345531463623047, -..."
9998,['027566056#Rites_d#c#initiation'],"[0.009930375963449478, -0.0309145525097847, -0..."


## From bibliographic records embeddings to RAMEAU concepts embeddings

In [19]:
df_data_with_vectors.rename(columns={'RAMEAU_concepts2': 'rameau_list'}, inplace=True)

In [20]:
df_data_with_vectors

,rameau_list,bibliographic_vector
0,"['027236005#Linguistique', '027789853#Étude_et...","[0.010903417132794857, 0.007470165845006704, -..."
1,"['027416593#Politique_culturelle', '027224929#...","[0.03385547548532486, -0.018181508406996727, 0..."
2,['027350282#Purges_politiques'],"[0.033818118274211884, -0.008515330962836742, ..."
3,"['029251133#Identité_collective', '027336360#D...","[0.019312692806124687, -0.00716855563223362, -..."
4,['027246086#Révoltes_paysannes'],"[0.015167579054832458, -0.025262555107474327, ..."
...,...,...
9995,"['031281737#Nombre_de_chromosomes', '027231429...","[0.010252579115331173, -0.007863505743443966, ..."
9996,"['027264998#Enseignement', '027264998#Enseigne...","[0.027775287628173828, -0.01775376684963703, -..."
9997,"['027264998#Enseignement', '027264998#Enseigne...","[0.03788097947835922, -0.009345531463623047, -..."
9998,['027566056#Rites_d#c#initiation'],"[0.009930375963449478, -0.0309145525097847, -0..."


In [21]:
# From bibliographic records embeddings to RAMEAU concepts embeddings
import ast
# Dataframe explosion : one RAMEAU concept per dataframe row
df_data_with_vectors["rameau_list"] = df_data_with_vectors["rameau_list"].apply(ast.literal_eval)
df_exploded = df_data_with_vectors.explode('rameau_list')
df_exploded.rename(columns={'rameau_list': 'rameau_concept'}, inplace=True)

# Grouping bibliographic records by RAMEAU concept
# RAMEAU concept embedding as mean of "its" bibliographic records embeddings
rameau_vectors = df_exploded.groupby('rameau_concept').agg(mean=('bibliographic_vector', lambda x: np.vstack(x).mean(axis=0).tolist()))

rameau_vectors['rameau_concept'] = rameau_vectors.index
rameau_vectors.rename(columns={'mean': 'rameau_vector'}, inplace=True)


In [22]:
rameau_vectors

,rameau_vector,rameau_concept
rameau_concept,,
02721818X#Vingt_et_unième_siècle,"[0.003659807611256838, -0.014624590054154396, ...",02721818X#Vingt_et_unième_siècle
027218244#Accumulateurs,"[0.018560584355145692, 0.007514845859259367, -...",027218244#Accumulateurs
027218260#Acoustique_architecturale,"[0.03547878935933113, 0.005090087652206421, -0...",027218260#Acoustique_architecturale
027218279#Actes_de_langage,"[0.010006211959989741, 0.006679423793684691, -...",027218279#Actes_de_langage
027218287#Acteurs_de_cinéma,"[0.005620877520414069, -0.002341044833883643, ...",027218287#Acteurs_de_cinéma
...,...,...
235634115#Diplodus,"[0.012709829490631819, -0.01425916375592351, -...",235634115#Diplodus
235976776#Et_les_autochtones,"[0.008877367712557316, -0.016923191491514444, ...",235976776#Et_les_autochtones
268200459#Dopage_Laser,"[0.02321726456284523, 0.012009331956505775, -0...",268200459#Dopage_Laser


In [23]:
rameau_vectors.to_pickle("RameauVectors_"+train_filename+".pkl")

## Storing RAMEAU concepts embeddings in a vector database

In [24]:
rameau_vectors = pd.read_pickle("RameauVectors_"+train_filename+".pkl")
rameau_vectors.head()

,rameau_vector,rameau_concept
rameau_concept,,
02721818X#Vingt_et_unième_siècle,"[0.003659807611256838, -0.014624590054154396, ...",02721818X#Vingt_et_unième_siècle
027218244#Accumulateurs,"[0.018560584355145692, 0.007514845859259367, -...",027218244#Accumulateurs
027218260#Acoustique_architecturale,"[0.03547878935933113, 0.005090087652206421, -0...",027218260#Acoustique_architecturale
027218279#Actes_de_langage,"[0.010006211959989741, 0.006679423793684691, -...",027218279#Actes_de_langage
027218287#Acteurs_de_cinéma,"[0.005620877520414069, -0.002341044833883643, ...",027218287#Acteurs_de_cinéma


In [ ]:
#qdrant.close()
qdrant = QdrantClient(path=qdrant_dbname) # create qdrant

# Create collection to store rameau vectors and labels (recreate = erases and recreates a collection with the same name)
qdrant.recreate_collection(
    collection_name=qdrant_collectionname,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

# upload to qdrant

qdrant.upload_records(
    collection_name=qdrant_collectionname,
    records=[
        models.Record(
            id=idx,
            vector=list(doc[0]),
            payload=dict.fromkeys( "aaa",doc[1])
        ) for idx, doc in enumerate(rameau_vectors.values)
    ]
)

qdrant.close()

## Predicting RAMEAU concepts from Title+Abstract

In [26]:

def predict(text, hits_count):
    hits = qdrant.search(
        collection_name=qdrant_collectionname,
        query_vector=encoder.encode(text).tolist(),
        limit=hits_count
    )
    load_items = []

    for hit in hits:
        load_items.append({'score': hit.score, 'label': hit.payload})

    return load_items


### Predicting subjects on one single document (title + summary)

In [27]:
#qdrant.close()

In [28]:
qdrant = QdrantClient(path=qdrant_dbname) # creating the qdrant client

text = "Les sommets de l'État : essai sur l'élite du pouvoir en France"

print(predict(text, 10))

qdrant.close()

[{'score': 0.8887674722028018, 'label': {'a': '027322610#Hauts_fonctionnaires'}}, {'score': 0.8876485029493538, 'label': {'a': '027229629#Bureaucratie'}}, {'score': 0.878010362000074, 'label': {'a': '027223345#Classes_dirigeantes'}}, {'score': 0.8708184005408419, 'label': {'a': '027994775#Institutions_politiques'}}, {'score': 0.8695460206723448, 'label': {'a': '027792102#Aspect_politique'}}, {'score': 0.8688430697309564, 'label': {'a': '027225224#Élite_(sciences_sociales)'}}, {'score': 0.868140065578612, 'label': {'a': '027365581#Pouvoir_(sciences_sociales)'}}, {'score': 0.8678212807323591, 'label': {'a': '02726470X#Histoire'}}, {'score': 0.8675603540922248, 'label': {'a': '027728110#Politique_et_gouvernement'}}, {'score': 0.8674706767092346, 'label': {'a': '027311163#Caractère_national_français'}}]


### Predicting subjects on the test dataset

In [29]:

start_time = datetime.now() 

qdrant = QdrantClient(path=qdrant_dbname) # creating the qdrant client

df_res = pd.DataFrame()

# Concatenating the 'TITRE' et 'RESUME' columns into the new column 'DESCR'
test_data["DESCR"] = test_data['TITRE'].astype(str) + ". " + test_data["RESUME"]


# Predicting + preparing the output table
prediction_results = []
for _, row in test_data.iterrows():
    predictions = predict(row["DESCR"], 6)  #     # Apply predict() to each prediction
    for p in predictions :
        extracted = {'ppn': row["PPN"], 'score': p['score'], 'label': p['label']['a']}
        prediction_results.append(extracted)

df_res = pd.DataFrame(prediction_results)

qdrant.close()

# Cleaning up the label column
df_res['label'] = df_res['label'].replace('#u#', '_', regex=True).replace('#d#', '"', regex=True).replace('#c#', "'", regex=True).replace('_', ' ', regex=True)

end_time = datetime.now() 
time_difference = (end_time - start_time).total_seconds() * 10**3
print("Execution time of program is: ", time_difference, "ms") 

df_res.to_csv("test_data_100_resu_" + train_filename + ".csv", index=False)
df_res.head(20)

Execution time of program is:  14365.581 ms


,ppn,score,label
0,000308838,0.935785,027322610#Hauts fonctionnaires
1,000308838,0.910755,027229629#Bureaucratie
2,000308838,0.906412,027223345#Classes dirigeantes
3,000308838,0.899408,027225224#Élite (sciences sociales)
4,000308838,0.894725,027233936#Hommes politiques
5,000308838,0.894134,027664228#Activité politique
6,00094758X,0.945272,027882691#Dollar américain
7,00094758X,0.925055,027467821#Finances internationales
8,00094758X,0.899961,028045904#Marché financier
9,00094758X,0.891732,027728382#Politique monétaire
